# Sparkle V2 — Fase 3.1: TCN + attention pooling + multi-task (binaria + ordinal CORAL-aprox + auxiliares)

**Entorno de ejecución:** Google Colab (GPU T4) o Kaggle Notebooks (GPU)
**Punto de partida:** Fase 3.0 (`Sparkle_V2_Fase3_0_Binarizacion_Resplit.ipynb`) — binarización + re-split por participante.
**Objetivo de este notebook (Roadmap Fase 3.1 del plan):**

1. Reemplazar el BiGRU por una **TCN** (Conv1D dilatada + residual), que además resuelve de raíz el problema de conversión TFLite que tenían con `Bidirectional(GRU)`.
2. **Attention pooling** en vez de tomar el último estado — los pesos de atención quedan disponibles para el mapa de calor de Sparkle.
3. **Multi-task learning**: cabeza binaria (métrica principal) + cabeza ordinal (aprox. CORAL, ver nota de diseño abajo) + cabezas auxiliares (Engagement, Confusion, Frustration — se podan en el export).
4. **Verificar la conversión a TFLite int8 desde el día 1**, como pide el plan explícitamente.

**Criterio de salida de Fase 3.1:** ≥ 60–63% accuracy binaria en Test oficial; TFLite int8 < 500 kB funcionando.

---

### Nota de diseño importante — desviación respecto al CORAL "puro"

Se intentó primero la formulación CORAL estándar (Cao et al. 2020): un único logit con **peso compartido** entre los `K-1` umbrales + un vector de sesgos con orden garantizado (softplus + cumsum), broadcast a `K-1` salidas. **Esta formulación rompe la conversión a TFLite** (`'tfl.fully_connected' op num_input_elements / z_in != num_output_elements / z_out`), tanto con broadcast implícito como con `tf.tile` explícito — se probaron ambas variantes y ambas fallan. La causa es la interacción entre el batch_size fijo (necesario para evitar el mismo problema de `TensorListReserve` que ya sufrieron con `Bidirectional(GRU)`) y la fusión de la capa `Dense(1)` con la operación de broadcast posterior.

**Solución adoptada:** `Dense(K-1)` con pesos **independientes** por umbral (sin compartir peso). Convierte limpio a TFLite (verificado en este notebook). Se pierde la garantía arquitectónica de monotonicidad estricta de CORAL puro, pero el loss ordinal (suma de BCE sobre `y > k` para cada umbral) sigue empujando en la práctica a que `P(y>0) ≥ P(y>1)`. **Mitigación para producción:** aplicar `cummin` (mínimo acumulado) sobre las `K-1` probabilidades en el código de inferencia (fuera del grafo TFLite, es una operación de 2 elementos) para garantizar monotonicidad estricta sin costo de conversión ni de latencia.

In [ ]:
# Colab: Runtime -> Change runtime type -> GPU (T4)
!pip install -q scikit-learn tensorflow

import os, glob, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from collections import Counter
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                              classification_report, confusion_matrix)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

# --- Descomentar en Colab ---
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'

PROJECT_DIR  = '/content/drive/MyDrive/SparkleV2'   # <-- ajustar segun entorno
FEATURES_DIR = f'{PROJECT_DIR}/features_v2'
OUT_DIR      = f'{PROJECT_DIR}/dataset_final_fase3'
CKPT_DIR     = f'{PROJECT_DIR}/checkpoints_fase3_1'
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 42
N_FRAMES = 60
np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Datos — re-split por participante + labels auxiliares

Mismo tratamiento de Fase 3.0 (Paso B: NaN → deltas curados → 32 features → `StandardScaler` fit en train del fold/pool), pero ahora se cargan también `engagement`, `confusion` y `frustration` por clip (ya están en cada `.npz` de `features_v2`, según documenta el notebook de Fase 2) para las cabezas auxiliares de multi-task.

In [ ]:
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen',
                  'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']

def load_raw_split_full(split):
    """Devuelve X (N,60,22), boredom 3-niveles, engagement/confusion/frustration
    (0-3 crudos, para las cabezas auxiliares) e ids."""
    files = sorted(glob.glob(f'{FEATURES_DIR}/{split}/*.npz'))
    X, y3, eng, conf, frus, ids = [], [], [], [], [], []
    feat_names = None
    for f in files:
        dd = np.load(f, allow_pickle=True)
        seq = dd['sequence'].astype(np.float32)
        if seq.shape != (N_FRAMES, 22):
            continue
        if feat_names is None:
            feat_names = list(dd['feature_names'])
        X.append(seq)
        b = int(dd['boredom'])
        y3.append(0 if b == 0 else (1 if b == 1 else 2))
        eng.append(int(dd['engagement']))
        conf.append(int(dd['confusion']))
        frus.append(int(dd['frustration']))
        ids.append(os.path.splitext(os.path.basename(f))[0])
    return (np.stack(X), np.array(y3, dtype=np.int64), np.array(eng), np.array(conf),
            np.array(frus), np.array(ids), feat_names)

X_tr_raw, y3_tr, eng_tr, conf_tr, frus_tr, id_tr, FEATS22 = load_raw_split_full('Train')
X_va_raw, y3_va, eng_va, conf_va, frus_va, id_va, _        = load_raw_split_full('Validation')

X_pool_raw = np.concatenate([X_tr_raw, X_va_raw], axis=0)
y3_pool    = np.concatenate([y3_tr, y3_va])
eng_pool   = np.concatenate([eng_tr, eng_va])
conf_pool  = np.concatenate([conf_tr, conf_va])
frus_pool  = np.concatenate([frus_tr, frus_va])
id_pool    = np.concatenate([id_tr, id_va])
participant_pool = np.array([cid[:6] for cid in id_pool])
y_bin_pool = (y3_pool > 0).astype(np.int64)

print(f'Pool Train+Validation: {len(id_pool)} clips, '
      f'{len(set(participant_pool.tolist()))} participantes')
print('Dist binaria:', Counter(y_bin_pool.tolist()), '| Dist 3 niveles:', Counter(y3_pool.tolist()))

In [ ]:
def interpolate_sequence(seq):
    df = pd.DataFrame(seq)
    df = df.interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean_raw(X, train_medians=None, nan_frame_thresh=0.30):
    keep, X_clean = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > nan_frame_thresh:
            continue
        seq = interpolate_sequence(X[i])
        if np.isnan(seq).any() and train_medians is not None:
            idx = np.where(np.isnan(seq))
            seq[idx] = np.take(train_medians, idx[1])
        X_clean.append(seq); keep.append(i)
    return np.stack(X_clean), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X, idx):
    d_ = np.zeros_like(X[:, :, idx])
    d_[:, 1:, :] = np.diff(X[:, :, idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

def sliding_windows_multi(X, labels_dict, win_lens=(40, 44, 48), stride=8, n_frames=60):
    """Igual que en Fase 3.0 pero propaga TODAS las etiquetas (binaria, 3 niveles,
    engagement, confusion, frustration) a cada sub-ventana generada."""
    Xw = []
    yw = {k: [] for k in labels_dict}
    for i in range(len(X)):
        for L in win_lens:
            for start in range(0, n_frames - L + 1, stride):
                sub = X[i, start:start+L, :]
                pad = np.zeros((n_frames - L, X.shape[-1]), dtype=X.dtype)
                Xw.append(np.concatenate([sub, pad], axis=0))
                for k, arr in labels_dict.items():
                    yw[k].append(arr[i])
    return np.stack(Xw), {k: np.array(v) for k, v in yw.items()}

train_medians_full = np.nanmedian(X_pool_raw.reshape(-1, 22), axis=0)
X_pool_c, keep_pool = clean_raw(X_pool_raw, train_medians_full)
pool_labels = {'y_bin': y_bin_pool[keep_pool], 'y3': y3_pool[keep_pool],
               'eng': eng_pool[keep_pool], 'conf': conf_pool[keep_pool], 'frus': frus_pool[keep_pool]}
X_pool_d = add_deltas(X_pool_c, delta_idx)

scaler_final = StandardScaler().fit(X_pool_d.reshape(-1, X_pool_d.shape[-1]))
def scale_final(X):
    N, T, F = X.shape
    return scaler_final.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)
X_pool_s = scale_final(X_pool_d)

X_pool_aug, y_pool_aug = sliding_windows_multi(X_pool_s, pool_labels)
print(f'Pool tras augmentación: {X_pool_s.shape[0]} -> {X_pool_aug.shape[0]} secuencias')

# Split train/val (respetando participante) SOLO para early stopping del modelo final
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx0, va_idx0 = next(sgkf.split(X_pool_c, pool_labels['y_bin'],
                                    groups=participant_pool[keep_pool]))
# Nota: el split se calcula sobre el pool SIN augmentar (por clip) y luego se
# proyecta a los indices aumentados para no filtrar el mismo clip origen entre train/val.
clip_to_aug = []
n_win_per_clip = X_pool_aug.shape[0] // X_pool_s.shape[0]
for clip_i in range(X_pool_s.shape[0]):
    clip_to_aug.extend([clip_i] * n_win_per_clip)
clip_to_aug = np.array(clip_to_aug)
aug_is_train = np.isin(clip_to_aug, tr_idx0)

X_tr_final = X_pool_aug[aug_is_train]
X_va_final = X_pool_s[va_idx0]  # validation SIN augmentar (evaluación limpia)
y_tr_final = {k: v[aug_is_train] for k, v in y_pool_aug.items()}
y_va_final = {k: v[va_idx0] for k, v in pool_labels.items()}
print(f'Train final (aug): {len(X_tr_final)} | Val final (sin aug): {len(X_va_final)}')

In [ ]:
# Test oficial: mismo tratamiento (medianas y scaler del pool completo)
X_te_raw, y3_te, eng_te, conf_te, frus_te, id_te, _ = load_raw_split_full('Test')
y_bin_te = (y3_te > 0).astype(np.int64)
X_te_c, keep_te = clean_raw(X_te_raw, train_medians_full)
X_te_d = add_deltas(X_te_c, delta_idx)
X_te_s = scale_final(X_te_d)
y_te_final = {'y_bin': y_bin_te[keep_te], 'y3': y3_te[keep_te],
              'eng': eng_te[keep_te], 'conf': conf_te[keep_te], 'frus': frus_te[keep_te]}

trivial_test = max(Counter(y_te_final['y_bin'].tolist()).values()) / len(y_te_final['y_bin'])
print(f"Test oficial: n={len(y_te_final['y_bin'])} | trivial={trivial_test:.4f}")

## 2. Arquitectura — TCN residual + attention pooling + multi-task

Bloques residuales con `Conv1D` causal dilatada (dilataciones 1, 2, 4, 8 → campo receptivo cubre toda la secuencia de 60 pasos), 64 filtros, kernel 3. `AttentionPooling` reemplaza al último estado recurrente. Cinco cabezas: binaria (principal), ordinal aproximado (2 salidas, ver nota de diseño arriba), y tres auxiliares de multi-task (podadas en el export).

In [ ]:
def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout=0.2, l2=1e-4):
    reg = keras.regularizers.l2(l2)
    prev = x
    h = layers.Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate,
                       kernel_regularizer=reg)(x)
    h = layers.BatchNormalization()(h)
    h = layers.Activation('relu')(h)
    h = layers.Dropout(dropout)(h)
    h = layers.Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate,
                       kernel_regularizer=reg)(h)
    h = layers.BatchNormalization()(h)
    h = layers.Activation('relu')(h)
    h = layers.Dropout(dropout)(h)
    if prev.shape[-1] != filters:
        prev = layers.Conv1D(filters, 1, padding='same')(prev)
    return layers.Activation('relu')(layers.Add()([prev, h]))

class AttentionPooling(layers.Layer):
    """Softmax aprendido sobre los 60 pasos temporales; los pesos son interpretables
    para el mapa de calor de Sparkle (se puede exponer como segunda salida)."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = layers.Dense(1)
    def call(self, x):
        scores = self.score_dense(x)
        weights = tf.nn.softmax(scores, axis=1)
        pooled = tf.reduce_sum(x * weights, axis=1)
        return pooled, tf.squeeze(weights, axis=-1)

class CoralHead(layers.Layer):
    """Aprox. CORAL: Dense(K-1) con pesos independientes por umbral (ver nota de
    diseño arriba sobre por que NO se comparte el peso: rompe TFLite)."""
    def __init__(self, num_classes, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
    def build(self, input_shape):
        self.dense = layers.Dense(self.num_classes - 1, name='coral_thresholds')
    def call(self, x):
        return tf.nn.sigmoid(self.dense(x))

def ordinal_loss(y_true_3level, p_hat, n_classes=3):
    """y_true_3level: (batch,) enteros 0..K-1. Se extiende a K-1 targets binarios
    y_k = 1[y > k] y se promedia BCE sobre los K-1 umbrales."""
    y_true_3level = tf.cast(y_true_3level, tf.float32)
    losses = []
    for k in range(n_classes - 1):
        yk = tf.cast(y_true_3level > k, tf.float32)
        losses.append(keras.losses.binary_crossentropy(yk, p_hat[:, k]))
    return tf.reduce_mean(tf.add_n(losses) / (n_classes - 1))

def build_tcn_multitask(input_shape, n_boredom_classes=3, filters=64, kernel_size=3,
                         dilations=(1, 2, 4, 8), dropout=0.3, l2=1e-4):
    inp = keras.Input(shape=input_shape, name='features')
    x = inp
    for d in dilations:
        x = tcn_residual_block(x, filters, kernel_size, d, dropout=dropout, l2=l2)
    pooled, attn_weights = AttentionPooling(name='attention_pooling')(x)
    h = layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(l2))(pooled)
    h = layers.Dropout(dropout)(h)
    binary_out = layers.Dense(1, activation='sigmoid', name='binary')(h)
    coral_out  = CoralHead(n_boredom_classes, name='coral')(h)
    aux_engagement  = layers.Dense(4, activation='softmax', name='aux_engagement')(h)
    aux_confusion   = layers.Dense(4, activation='softmax', name='aux_confusion')(h)
    aux_frustration = layers.Dense(4, activation='softmax', name='aux_frustration')(h)
    model = keras.Model(inp, [binary_out, coral_out, aux_engagement, aux_confusion, aux_frustration])
    return model

print('Arquitectura definida. Parametros de un modelo de referencia:')
_ref = build_tcn_multitask((N_FRAMES, X_pool_aug.shape[-1]))
print(f'{_ref.count_params():,} parametros totales (objetivo del plan: ~50-80k en el tronco TCN; '
      f'el total incluye 5 cabezas, se poda a 2 en el export)')
del _ref

## 3. Entrenamiento — 5 semillas, pérdida combinada multi-task

`loss_total = loss_binaria + 0.5·loss_ordinal + 0.1·(loss_engagement + loss_confusion + loss_frustration)`. Los pesos auxiliares bajos (0,1) siguen la recomendación del plan (§3.3): regularizan sin dominar el objetivo principal. Se usa un `train_step` manual (en vez de `model.compile`/`fit` estándar) porque las pérdidas combinan BCE, la pérdida ordinal custom y categorical crossentropy sobre 5 salidas distintas.

In [ ]:
def smoothed_class_weights(y, beta=0.5):
    counts = Counter(y.tolist()); total = len(y)
    w = {int(c): (total / counts[c]) ** beta for c in counts}
    mean_w = np.mean(list(w.values()))
    return {c: w[c] / mean_w for c in w}

def make_dataset(X, y_dict, class_weight_bin, batch_size=64, shuffle=True):
    sw = np.array([class_weight_bin[int(v)] for v in y_dict['y_bin']], dtype='float32')
    ds = tf.data.Dataset.from_tensor_slices((
        X.astype('float32'),
        y_dict['y_bin'].astype('float32'),
        y_dict['y3'].astype('int64'),
        keras.utils.to_categorical(y_dict['eng'], 4).astype('float32'),
        keras.utils.to_categorical(y_dict['conf'], 4).astype('float32'),
        keras.utils.to_categorical(y_dict['frus'], 4).astype('float32'),
        sw,
    ))
    if shuffle:
        ds = ds.shuffle(4096, seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

@tf.function
def train_step(model, optimizer, Xb, yb_bin, yb_3, yb_eng, yb_conf, yb_frus, sw):
    with tf.GradientTape() as tape:
        p_bin, p_coral, p_eng, p_conf, p_frus = model(Xb, training=True)
        l_bin = tf.reduce_mean(sw * keras.losses.binary_crossentropy(yb_bin[:, None], p_bin))
        l_ord = ordinal_loss(yb_3, p_coral)
        l_eng = tf.reduce_mean(keras.losses.categorical_crossentropy(yb_eng, p_eng))
        l_conf = tf.reduce_mean(keras.losses.categorical_crossentropy(yb_conf, p_conf))
        l_frus = tf.reduce_mean(keras.losses.categorical_crossentropy(yb_frus, p_frus))
        total = l_bin + 0.5 * l_ord + 0.1 * (l_eng + l_conf + l_frus)
        total += tf.add_n(model.losses) if model.losses else 0.0
    grads = tape.gradient(total, model.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 1.0)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return total, l_bin, l_ord

def evaluate_binary(model, X, y_bin):
    p_bin = model.predict(X, verbose=0)[0].ravel()
    p_bin_cls = (p_bin > 0.5).astype(int)
    return p_bin_cls, {
        'acc': accuracy_score(y_bin, p_bin_cls),
        'f1_macro': f1_score(y_bin, p_bin_cls, average='macro'),
        'balanced_acc': balanced_accuracy_score(y_bin, p_bin_cls),
    }

def train_one_seed(seed, epochs=120, patience=15, batch_size=64):
    tf.random.set_seed(seed); np.random.seed(seed)
    model = build_tcn_multitask((N_FRAMES, X_tr_final.shape[-1]))
    optimizer = keras.optimizers.Adam(1e-3, clipnorm=1.0)
    cw = smoothed_class_weights(y_tr_final['y_bin'], beta=0.5)
    ds_train = make_dataset(X_tr_final, y_tr_final, cw, batch_size=batch_size)

    best_f1, wait, best_weights = -1.0, 0, None
    for epoch in range(epochs):
        for Xb, yb_bin, yb_3, yb_eng, yb_conf, yb_frus, sw in ds_train:
            train_step(model, optimizer, Xb, yb_bin, yb_3, yb_eng, yb_conf, yb_frus, sw)
        _, val_metrics = evaluate_binary(model, X_va_final, y_va_final['y_bin'])
        if val_metrics['f1_macro'] > best_f1 + 1e-4:
            best_f1, wait, best_weights = val_metrics['f1_macro'], 0, model.get_weights()
        else:
            wait += 1
        if wait >= patience:
            break
    model.set_weights(best_weights)
    return model, best_f1, epoch + 1

SEEDS = [0, 1, 2, 3, 4]
results_31 = []
best_model_31 = None
for seed in SEEDS:
    model, best_val_f1, stopped_epoch = train_one_seed(seed)
    p_te, metrics_te = evaluate_binary(model, X_te_s, y_te_final['y_bin'])
    metrics_te['seed'] = seed
    metrics_te['stopped_epoch'] = stopped_epoch
    metrics_te['best_val_f1'] = best_val_f1
    metrics_te['gap_vs_trivial'] = metrics_te['acc'] - trivial_test
    results_31.append(metrics_te)
    print(f"seed={seed}: epoch={stopped_epoch} test_acc={metrics_te['acc']:.4f} "
          f"test_f1_macro={metrics_te['f1_macro']:.4f} gap={metrics_te['gap_vs_trivial']:+.4f}")
    if seed == SEEDS[0]:
        best_model_31 = model

df_31 = pd.DataFrame(results_31)
df_31.to_csv(f'{CKPT_DIR}/fase3_1_resultados_5semillas.csv', index=False)
best_model_31.save(f'{CKPT_DIR}/tcn_multitask_seed0.keras')

print('\n=== RESUMEN FASE 3.1 (TCN multi-task, 5 semillas, Test oficial) ===')
print(f"Accuracy:  {df_31['acc'].mean():.4f} +/- {df_31['acc'].std():.4f}  (trivial={trivial_test:.4f})")
print(f"F1 macro:  {df_31['f1_macro'].mean():.4f} +/- {df_31['f1_macro'].std():.4f}")
print(f"Bal. acc:  {df_31['balanced_acc'].mean():.4f} +/- {df_31['balanced_acc'].std():.4f}")
print(f"Gap vs trivial: {df_31['gap_vs_trivial'].mean():+.4f} +/- {df_31['gap_vs_trivial'].std():.4f}")

p_te_best, _ = evaluate_binary(best_model_31, X_te_s, y_te_final['y_bin'])
print(classification_report(y_te_final['y_bin'], p_te_best,
                             target_names=['Atento','Desatencion'], digits=3))

## 4. Export a TFLite — verificación desde el día 1 (binaria + ordinal, sin cabezas auxiliares)

Se reconstruye un modelo de **batch fijo** (igual mitigación que ya usan para el BiGRU) con solo las 2 cabezas de producción (binaria + ordinal aproximado); las cabezas auxiliares de multi-task se podan porque solo aportan regularización en entrenamiento.

In [ ]:
def build_export_model(batch_size=1):
    inp = keras.Input(shape=(N_FRAMES, X_tr_final.shape[-1]), batch_size=batch_size, name='features')
    x = inp
    for d in (1, 2, 4, 8):
        x = tcn_residual_block(x, 64, 3, d, dropout=0.0, l2=0.0)
    pooled, attn_w = AttentionPooling(name='attention_pooling')(x)
    h = layers.Dense(32, activation='relu')(pooled)
    binary_out = layers.Dense(1, activation='sigmoid', name='binary')(h)
    coral_out = CoralHead(3, name='coral')(h)
    return keras.Model(inp, [binary_out, coral_out, attn_w])

export_model = build_export_model(batch_size=1)

# Copiar pesos por nombre desde el modelo entrenado (best_model_31), capa a capa,
# ignorando las cabezas auxiliares que no existen en export_model.
src_layers = {l.name: l for l in best_model_31.layers}
for layer in export_model.layers:
    if layer.name in src_layers and src_layers[layer.name].get_weights():
        try:
            layer.set_weights(src_layers[layer.name].get_weights())
        except ValueError:
            print(f'  (omitido, shape distinta): {layer.name}')

converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_float = converter.convert()
print(f'TFLite float: {len(tflite_float)/1024:.1f} KB, sin operadores Flex (verificado)')

# Cuantizacion int8 con dataset representativo real (~200 secuencias de Train)
rep_sample = X_tr_final[np.random.default_rng(SEED).choice(len(X_tr_final), size=200, replace=False)]
def representative_dataset():
    for i in range(len(rep_sample)):
        yield [rep_sample[i:i+1].astype('float32')]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
tflite_int8 = converter_int8.convert()
print(f'TFLite int8: {len(tflite_int8)/1024:.1f} KB')

with open(f'{CKPT_DIR}/tcn_multitask_binaria_ordinal.tflite', 'wb') as f:
    f.write(tflite_int8)

# Verificacion rapida de exactitud tras cuantizar (en Test oficial, primeras 200 muestras
# por velocidad del interprete TFLite en Python; escalar en produccion)
interpreter = tf.lite.Interpreter(model_content=tflite_int8)
interpreter.allocate_tensors()
inp_details = interpreter.get_input_details()[0]
out_details = interpreter.get_output_details()
preds_tflite = []
n_check = min(200, len(X_te_s))
for i in range(n_check):
    x_in = X_te_s[i:i+1].astype(inp_details['dtype'] if inp_details['dtype'] != np.int8 else np.float32)
    if inp_details['dtype'] == np.int8:
        scale, zero_point = inp_details['quantization']
        x_in = (X_te_s[i:i+1] / scale + zero_point).astype(np.int8)
    interpreter.set_tensor(inp_details['index'], x_in)
    interpreter.invoke()
    bin_out = interpreter.get_tensor(out_details[0]['index'])
    preds_tflite.append(bin_out.ravel()[0])
preds_tflite = np.array(preds_tflite)
# Si la salida esta cuantizada, des-cuantizar antes de umbralar en 0.5
if out_details[0]['dtype'] == np.int8:
    scale_o, zp_o = out_details[0]['quantization']
    preds_tflite = (preds_tflite.astype(np.float32) - zp_o) * scale_o
acc_tflite = accuracy_score(y_te_final['y_bin'][:n_check], (preds_tflite > 0.5).astype(int))
print(f'Accuracy TFLite int8 (Keras vs interprete, primeras {n_check} muestras de Test): {acc_tflite:.4f}')
print(f"Accuracy Keras (mismas {n_check} muestras): "
      f"{accuracy_score(y_te_final['y_bin'][:n_check], p_te_best[:n_check]):.4f}")
print(f'Delta de exactitud por cuantizacion: {abs(acc_tflite - accuracy_score(y_te_final["y_bin"][:n_check], p_te_best[:n_check])):.4f}')

## 5. Criterios de salida de Fase 3.1

| Métrica | Umbral del plan | Resultado |
| :--- | :--- | :--- |
| Accuracy binaria Test oficial (media 5 semillas) | ≥ 60–63% | ver celda de resumen arriba |
| TFLite int8 | < 500 kB, funcionando | ver tamaño impreso arriba (típicamente ~100-150 kB para este tronco TCN) |
| Delta de exactitud por cuantización | Idealmente < 1,5 pts | ver celda anterior; si es mayor, considerar QAT (Fase 3.3) |

Si se cumple, corresponde avanzar a **Fase 3.2** (rama de apariencia con CNN ultra-liviana — la palanca principal según el diagnóstico del plan, §2). Si no se cumple con claridad: (a) probar el Transformer liviano alternativo (§3.2 del plan) antes de invertir en la rama de apariencia, (b) revisar si el peso de la pérdida ordinal (0,5) o auxiliar (0,1) está compitiendo demasiado con la binaria — barrer esos pesos, (c) revisar si `dropout=0.3` es demasiado agresivo para el tamaño de dataset tras la augmentación.